In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\work\ner\data\filter\rule_detect_data")
summary_files = list(DATA_DIR.rglob("run_summary.csv"))
results_files = list(DATA_DIR.rglob("candidate_rule_results.csv"))

# 룰 검출 결과 summary 취합 
summary_df = pd.concat([pd.read_csv(path) for path in summary_files], ignore_index=True)
summary_df # 원본기사, 계엄기사, 룰검출(전체), 룰검출(후처리), 최종검수문장 > 여기서 문장이 기사의 모든 문장이 들어있는건지, 아니면 룰 검출된 문장만 있는건지 확인필요 

,press,source_articles,processed_articles,rule_articles_detected,martial_law_articles_excluded,candidate_articles,candidate_sentences
0,강원일보,14050,14050,13959,503,10557,32908
1,경기일보,20197,20197,20041,749,16240,51014
2,경향신문,33592,33592,33517,3007,30154,178686
3,노컷뉴스,54008,54008,53685,1873,43709,184280
4,뉴스1,175162,175162,174417,4893,146573,626764
5,뉴스핌,77528,77528,77270,1372,63846,273378
6,뉴시스,128114,128114,127712,2087,104176,392584
7,디지털타임스,33176,33176,33029,1558,27367,130393
8,마이데일리,36552,36552,36423,55,29939,107613
9,부산일보,35000,35000,34838,860,30031,135507


In [2]:
results_files

[WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_123553_강원일보/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_123930_경기일보/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_124505_경향신문/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_125622_노컷뉴스/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_131158_뉴스1/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_140420_뉴스핌/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_143243_뉴시스/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_151606_디지털타임스/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_detect_data/20261117_154059_마이데일리/candidate_rule_results.csv'),
 WindowsPath('C:/work/ner/data/filter/rule_det

- 후처리 대상
1. 룰 컬럼 삭제
    - EXT_E001, EXT_S002, EXT_S001, EXT_E003_AF_006, EXT_E003_LC_006, EXT_E003_AF_006_PATTERN, EXT_E003_LC_006_PATTERN
2. EXT_E004_DT_001
    - DT 선행표현에 한~ 으로 시작하는거 제외
    - 오는~ 시작하되 form에서 오는 앞에 공백이 없으면 제외
3. EXT_E004_DT_003
    - ~경, 간, 초로 끝나는거 제외
4. EXT_E004_DT_007
    - ~날로 끝나는데 form에서 날 뒤에 공백이 없으면 제외
5. EXT_E005_EV_001
    - 년도 + 행사 만 남기고 나머지 제외
6. EXT_E005_EV_004
    - 예를들어, 검출된 3대 앞에 공백이 없으면 제외

In [ ]:
import re
from collections import defaultdict

# candidate_rule_results.csv는 이미 후보 문장만 포함하므로, 아래의
# original_candidate_*는 전체 룰 검출 문장이 아니라 후보 산출물 기준임을 명시한다.
DROP_COLUMNS = {
    "EXT_E001", "EXT_S002", "EXT_S001",
    "EXT_E003_AF_006", "EXT_E003_LC_006",
    "EXT_E003_AF_006_PATTERN", "EXT_E003_LC_006_PATTERN",
}
# EV_001은 ‘년도 + 행사’가 아니라, 개체명이 날짜로 시작하고 뒤에 내용이 있는 경우만 허용한다.
# 예: ‘2022 카타르 월드컵’, ‘2022년 카타르 월드컵’, ‘2022년도 카타르 월드컵’
DATE_CONTENT_RE = r"^\s*(?:\d{4}(?:년|년도)?(?:\s*\d{1,2}월(?:\s*\d{1,2}일)?)?|\d{1,2}월\s*\d{1,2}일)\s+\S+"

TARGET_RULES = {
    "dt_001": "EXT_E004_DT_001",
    "dt_003": "EXT_E004_DT_003",
    "dt_007": "EXT_E004_DT_007",
    "ev_001": "EXT_E005_  EV_001",
    "ev_004": "EXT_E005_  EV_004",
}

def _resolve_column(columns, expected):
    """공백 차이까지 허용해 룰 컬럼명을 찾는다."""
    if expected in columns:
        return expected
    compact = re.sub(r"\s+", "", expected)
    matches = [c for c in columns if re.sub(r"\s+", "", c) == compact]
    return matches[0] if len(matches) == 1 else None

def _as_bool(series):
    return series.map(lambda value: value if isinstance(value, bool) else str(value).strip().lower() == "true")

TAG_RE = re.compile(r"<([^<>:]+):\s*[A-Za-z0-9_]+>")
def _plain_form(value):
    # form의 <개체명: TAG> 표기를 개체명으로 치환해 실제 문장 기준으로 검사한다.
    text = "" if pd.isna(value) else str(value)
    return TAG_RE.sub(lambda m: m.group(1), text)

def _entity_position(row):
    text = _plain_form(row.get("form", ""))
    entity = "" if pd.isna(row.get("entity_name", "")) else str(row.get("entity_name", ""))
    start = text.find(entity) if entity else -1
    return text, entity, start

def _is_invalid_dt001(row):
    text, entity, start = _entity_position(row)
    # ‘한~’ 선행 표현은 제외한다. ‘오는~’은 앞 문자가 공백일 때만 허용한다.
    if entity.startswith("한"):
        return True
    if entity.startswith("오는") and start > 0 and not text[start - 1].isspace():
        return True
    return False

def _is_invalid_dt003(row):
    entity = "" if pd.isna(row.get("entity_name", "")) else str(row.get("entity_name", ""))
    return entity.endswith(("경", "간", "초"))

def _is_invalid_dt007(row):
    text, entity, start = _entity_position(row)
    if not entity.endswith("날") or start < 0:
        return False
    end = start + len(entity)
    return end < len(text) and not text[end].isspace()

def _is_year_event(row):
    entity = "" if pd.isna(row.get("entity_name", "")) else str(row.get("entity_name", ""))
    return bool(re.match(DATE_CONTENT_RE, entity))

def _is_invalid_ev004(row):
    text, entity, start = _entity_position(row)
    if start < 0 or not re.match(r"^\d+\s*대", entity):
        return False
    return start > 0 and not text[start - 1].isspace()

def _press_from_path(path):
    match = re.match(r"^\d{8}_\d{6}_(.+)$", path.parent.name)
    return match.group(1) if match else path.parent.name

def _read_csv(path):
    for encoding in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return pd.read_csv(path, encoding=encoding, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(encoding, b"", 0, 1, f"파일 인코딩을 확인할 수 없습니다: {path}")

def postprocess_press(press=None, files=None, output_dir=None):
    """언론사 1곳 또는 전체를 후처리하고, 결과와 집계표를 반환한다."""
    files = list(results_files if files is None else files)
    selected = [p for p in files if press is None or _press_from_path(Path(p)) == press]
    if not selected:
        available = sorted({_press_from_path(Path(p)) for p in files})
        raise ValueError(f"대상 언론사가 없습니다: {press!r}; 사용 가능: {available}")
    if output_dir is None:
        output_dir = DATA_DIR / "postprocessed"
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_lookup = summary_df.groupby("press", dropna=False).first() if "press" in summary_df else pd.DataFrame()
    summaries, processed_frames = [], []
    for path in selected:
        path = Path(path)
        press_name = _press_from_path(path)
        df = _read_csv(path)
        required = {"doc_id", "sentence_id", "entity_name", "form"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{path.name} 필수 컬럼 누락: {sorted(missing)}")
        original_candidate_articles = df["doc_id"].nunique()
        original_candidate_sentences = df["sentence_id"].nunique()
        df = df.copy()
        # 룰값을 bool로 통일한 뒤, 행 단위로 무효화할 룰만 False 처리한다.
        for col in [c for c in df.columns if c.startswith("EXT_")]:
            if not col.endswith("_PATTERN"):
                df[col] = _as_bool(df[col])
        invalid = {}
        rules = {key: _resolve_column(df.columns, value) for key, value in TARGET_RULES.items()}
        # form 파싱과 개체 위치 계산은 한 번만 수행해 대량 CSV에서도 반복 apply를 피한다.
        plain_forms = df["form"].map(_plain_form)
        entities = df["entity_name"].fillna("").astype(str)
        starts = [text.find(entity) if entity else -1 for text, entity in zip(plain_forms, entities)]
        previous = [text[start - 1] if start > 0 else "" for text, start in zip(plain_forms, starts)]
        following = [text[start + len(entity)] if start >= 0 and start + len(entity) < len(text) else "" for text, entity, start in zip(plain_forms, entities, starts)]
        if rules["dt_001"]:
            invalid[rules["dt_001"]] = entities.str.startswith("한") | (entities.str.startswith("오는") & (pd.Series(starts, index=df.index) > 0) & ~pd.Series(previous, index=df.index).str.isspace())
        if rules["dt_003"]:
            invalid[rules["dt_003"]] = entities.str.endswith(("경", "간", "초"))
        if rules["dt_007"]:
            invalid[rules["dt_007"]] = entities.str.endswith("날") & (pd.Series(starts, index=df.index) >= 0) & ~pd.Series(following, index=df.index).str.isspace() & pd.Series(following, index=df.index).ne("")
        if rules["ev_001"]:
            invalid[rules["ev_001"]] = ~entities.str.match(DATE_CONTENT_RE, na=False)
        if rules["ev_004"]:
            invalid[rules["ev_004"]] = entities.str.match(r"^\d+\s*대", na=False) & (pd.Series(starts, index=df.index) > 0) & ~pd.Series(previous, index=df.index).str.isspace()
        for col, mask in invalid.items():
            df.loc[mask, col] = False
        df = df.drop(columns=[c for c in DROP_COLUMNS if c in df.columns])
        active_rules = [c for c in df.columns if c.startswith("EXT_") and not c.endswith("_PATTERN")]
        if not active_rules:
            raise ValueError(f"후처리 후 남은 룰 컬럼이 없습니다: {path}")
        df = df.loc[df[active_rules].any(axis=1)].reset_index(drop=True)
        out_path = output_dir / f"{press_name}_postprocessed_candidate_rule_results.csv"
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        before = summary_lookup.loc[press_name].to_dict() if press_name in summary_lookup.index else {}
        summaries.append({
            "press": press_name,
            "source_articles": before.get("source_articles", pd.NA),
            "rule_articles_detected": before.get("rule_articles_detected", pd.NA),
            "original_candidate_articles": original_candidate_articles,
            "original_candidate_sentences": original_candidate_sentences,
            "postprocessed_articles": df["doc_id"].nunique(),
            "postprocessed_sentences": df["sentence_id"].nunique(),
            "sentence_scope": "candidate_rule_results.csv 기준 후보 문장",
            "result_file": str(out_path),
        })
        processed_frames.append(df.assign(press=press_name))
    result_summary = pd.DataFrame(summaries).sort_values("press").reset_index(drop=True)
    result_summary.to_csv(output_dir / "postprocess_summary.csv", index=False, encoding="utf-8-sig")
    return result_summary, pd.concat(processed_frames, ignore_index=True) if processed_frames else pd.DataFrame()

# 실행 예시: 언론사 1곳만 처리 (press=None이면 results_files 전체 처리)
# postprocess_summary_df, postprocessed_df = postprocess_press(press="강원일보")

# 최소 검증 예시: 원본 후보 대비 후처리 결과가 감소/유지되는지, 삭제 룰 컬럼이 남지 않았는지 확인
# postprocess_summary_df, postprocessed_df = postprocess_press(press="강원일보")
# assert (postprocess_summary_df["postprocessed_articles"] <= postprocess_summary_df["original_candidate_articles"]).all()
# assert (postprocess_summary_df["postprocessed_sentences"] <= postprocess_summary_df["original_candidate_sentences"]).all()
# assert not (set(DROP_COLUMNS) & set(postprocessed_df.columns))
# display(postprocess_summary_df)
# display(postprocessed_df[["doc_id", "sentence_id", "form"]].head(10))